## 1. Initialize Project Environment
Import dependencies, configure Entrez email, and set up output paths.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Dict, List

from Bio import Entrez

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

biopython 1.85


## 2. Define Configuration Parameters
Centralize query settings, handle, and output paths for reproducibility.

In [2]:
from dataclasses import dataclass, asdict


@dataclass
class PubMedConfig:
    handle: str
    query: str
    max_results: int = 5
    email: str = "student@example.com"
    export_dir: Path = Path("artifacts")

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = PubMedConfig(
    handle="AndreiCod",
    query="TP53 AND cancer",
    max_results=5,
)

CONFIG.describe()

{'handle': 'AndreiCod',
 'query': 'TP53 AND cancer',
 'max_results': 5,
 'email': 'student@example.com',
 'export_dir': 'artifacts'}

## 3. Implement Core Functionality
Build utilities for searching PubMed and fetching article details.

In [3]:
def search_pubmed(query: str, max_results: int, email: str) -> List[str]:
    """Search PubMed and return list of PMIDs."""
    Entrez.email = email
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    return record["IdList"]


def fetch_article_details(pmid: str, email: str) -> Dict:
    """Fetch article details (title, authors, abstract) for a given PMID."""
    Entrez.email = email
    handle = Entrez.efetch(db="pubmed", id=pmid, rettype="xml", retmode="xml")
    records = Entrez.read(handle)
    handle.close()

    article_data = {"pmid": pmid, "title": "", "authors": [], "abstract": ""}

    if records["PubmedArticle"]:
        article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]

        # Title
        article_data["title"] = article.get("ArticleTitle", "No title available")

        # Authors
        if "AuthorList" in article:
            for author in article["AuthorList"]:
                if "LastName" in author and "ForeName" in author:
                    article_data["authors"].append(
                        f"{author['LastName']} {author['ForeName']}"
                    )
                elif "LastName" in author:
                    article_data["authors"].append(author["LastName"])

        # Abstract
        if "Abstract" in article and "AbstractText" in article["Abstract"]:
            abstract_parts = article["Abstract"]["AbstractText"]
            if isinstance(abstract_parts, list):
                article_data["abstract"] = " ".join(
                    str(part) for part in abstract_parts
                )
            else:
                article_data["abstract"] = str(abstract_parts)

    return article_data


# Test search
pmids = search_pubmed(CONFIG.query, CONFIG.max_results, CONFIG.email)
print(f"Found {len(pmids)} articles: {pmids}")

Found 5 articles: ['41466633', '41466059', '41465586', '41465506', '41465396']


In [4]:
# Fetch details for all articles
articles = []
for i, pmid in enumerate(pmids, 1):
    print(f"Fetching article {i}/{len(pmids)}: PMID {pmid}")
    details = fetch_article_details(pmid, CONFIG.email)
    articles.append(details)

# Display first article
articles[0] if articles else "No articles found"

Fetching article 1/5: PMID 41466633
Fetching article 2/5: PMID 41466059
Fetching article 3/5: PMID 41465586
Fetching article 4/5: PMID 41465506
Fetching article 5/5: PMID 41465396


{'pmid': '41466633',
 'title': 'Genetic and epigenetic alterations in oral potentially malignant disorders: A cross-sectional clinical study.',
 'authors': ['Baheti Akanksha',
  'Mansukhbhai Timbadiya Vijaykumar',
  'M Raviya Parth',
  'Shilu Kajal',
  'Singh Jyoti',
  'Guruprasad Yadavalli'],
 'abstract': 'The prevalence and spectrum of genetic and epigenetic alterations in oral potentially malignant disorders (OPMDs) is of interest. Hence, a total of 132 patients with clinically and histopathologically diagnosed OPMDs were evaluated for key molecular changes, including TP53 mutations, promoter methylation of tumor suppressor genes and global DNA hypomethylation. Salivary and tissue samples were analyzed using PCR, methylation-specific PCR (MSP) and immunohistochemistry. TP53 mutations and p16INK4a promoter hypermethylation were significantly associated with severe dysplasia and higher malignant transformation risk. Thus, integrating molecular profiling into OPMD evaluation could impr

## 4. Validate with Unit Tests
Quick inline assertions to ensure the PubMed helpers work correctly.

In [5]:
def test_search_pubmed():
    results = search_pubmed("TP53", 2, CONFIG.email)
    assert len(results) <= 2
    assert all(r.isdigit() for r in results)


def test_fetch_article_details():
    if pmids:
        details = fetch_article_details(pmids[0], CONFIG.email)
        assert "pmid" in details
        assert "title" in details


test_search_pubmed()
test_fetch_article_details()
print("All inline tests passed.")

All inline tests passed.


## 5. Export Results
Save the PubMed search results to a text file.

In [6]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

out_file = EXPORT_DIR / "task1_pubmed_results.txt"

with open(out_file, "w", encoding="utf-8") as f:
    f.write(f"PubMed Query: {CONFIG.query}\n")
    f.write(f"Max Results: {CONFIG.max_results}\n")
    f.write("=" * 80 + "\n\n")

    for i, details in enumerate(articles, 1):
        f.write(f"Article {i}\n")
        f.write("-" * 40 + "\n")
        f.write(f"PMID: {details['pmid']}\n")
        f.write(f"Title: {details['title']}\n")
        f.write(f"Authors: {', '.join(details['authors'][:5])}")
        if len(details["authors"]) > 5:
            f.write(f" et al. ({len(details['authors'])} total)")
        f.write("\n")
        f.write(f"Abstract:\n{details['abstract']}\n\n")
        f.write("=" * 80 + "\n\n")

print(f"[OK] PubMed results saved to: {out_file.resolve()}")

[OK] PubMed results saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/03_formats&NGS/assignments/artifacts/task1_pubmed_results.txt
